# Performance Tracking During Training

This notebook analyzes the training progress of offline, fine-tuned, and online policies.  
It extracts the average reward collected at each evaluation checkpoint and visualizes the learning curves over time.

## Import libraries

Import libraries for file management, data processing, and plotting.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

## Configuration

Define the experiment type, list of tasks, algorithms, and smoothing window for the plots.

In [ ]:
experiment = 'offline'   # 'offline', 'finetuning', or 'online'
tasks = ['relocate', 'door', 'pen', 'hammer']
algorithms = ['iql', 'cql', 'bc', 'td3bc', 'awac']

smoothing_window = 5     # Smoothing window for the moving average (typically 3–10)

## Folder renaming

Normalize subfolder names to standard algorithm labels for consistent access.

In [ ]:
# Root directory containing training logs
log_dir = "training_logs"

# Traverse all folders within the training log directory
for run_folder in os.listdir(log_dir):
    run_path = os.path.join(log_dir, run_folder)
    
    # Skip hidden files or non-directory entries
    if os.path.isdir(run_path) and not run_folder.startswith('.'):
        for task in tasks:
            task_path = os.path.join(run_path, task)
            
            # Check if the task directory exists
            if os.path.isdir(task_path):
                for subfolder in os.listdir(task_path):
                    full_path = os.path.join(task_path, subfolder)
                    
                    # Look for algorithm folders with "_" in their name
                    if os.path.isdir(full_path) and "_" in subfolder:
                        base_name = subfolder.split("_")[0]
                        
                        # Normalize TD3PlusBC naming
                        if base_name == "TD3PlusBC":
                            base_name = "TD3BC"
                            
                        base_name = base_name.lower()
                        new_path = os.path.join(task_path, base_name)
                        
                        # Rename folder if target name doesn't already exist
                        if not os.path.exists(new_path):
                            os.rename(full_path, new_path)
                            print(f"Renamed: {full_path} → {new_path}")
                        else:
                            print(f"Skipped (already exists): {new_path}")

## Output folder

Create a folder to store performances.

In [ ]:
# Create output directory for saving performance plots
path = os.path.join("performance", experiment)

if not os.path.exists(path):
    os.makedirs(path)
    print(f"Created: {path}")
else:
    print(f"Already exists: {path}")

## Plot colors

Define color scheme for each algorithm in the graph:

In [ ]:
# Assign a specific color to each algorithm
colors = {
    'iql': 'tab:blue',
    'cql': 'tab:purple',
    'bc': 'tab:green',
    'td3bc': 'tab:red',
    'awac': 'tab:orange'
}

## Graph generation

Generate one graph per task, visualizing the average total reward per episode over time for each algorithm.

In [ ]:
# Select the appropriate filename depending on the experiment type
if experiment == 'online':
    filename = 'rollout_return'
else:
    filename = 'evaluation'

# Generate a training curve for each task
for task in tasks:
    plt.figure(figsize=(10, 6))

    for algo in algorithms:
        # Load reward logs
        df = pd.read_csv(f'training_logs/{experiment}/{task}/{algo}/{filename}.csv', header=None)
        df.columns = ['epoch', 'step', 'return']

        # Apply moving average smoothing
        df['smoothed_return'] = df['return'].rolling(window=smoothing_window, min_periods=1).mean()

        # Plot smoothed returns
        plt.plot(df['step'], df['smoothed_return'], label=algo, color=colors[algo], linewidth=2.0)

    # Set plot labels and formatting
    plt.xlabel("Training Step")
    plt.ylabel("Average Return")
    plt.title(f"Reward during training - {experiment} - {task}")
    plt.grid(True, axis='y')
    plt.legend(loc='upper left')
    plt.tight_layout()

    # Save plot to file
    plt.savefig(f'performance/{experiment}/{task}.png', dpi=300, bbox_inches='tight')
    plt.show()
